Execute para verificar se todos os arquivos hashes contidos no arquivo log md5 .zip do virusshare possuem correspondentes nas tabelas dos arquivos analisados
Os arquivos que estiverem faltando serão baixados utilizando o VirusShare-Search disponível em https://github.com/AdamGreenhill/VirusShare-Search

In [20]:
import csv
import sys

pack_number = "00469"

# Função para extrair o hash da coluna 'Arquivo'
def extrair_hash(arquivo_nome):
    prefixo = 'VirusShare_'
    if arquivo_nome.startswith(prefixo):
        return arquivo_nome[len(prefixo):].lower()
    return None

try:
    # Carregar os hashes MD5 em um conjunto para busca eficiente
    with open(f'md5/VirusShare_{pack_number}.md5', 'r') as f:
        # Supondo que cada linha contenha apenas o hash
        md5_hashes = set(line.strip().lower() for line in f if line.strip())
except FileNotFoundError:
    print(f"Erro: Arquivo 'md5/VirusShare_{pack_number}.md5' não encontrado.")
    sys.exit(1)

csv_hashes = set()

try:
    # Ler o arquivo CSV e extrair os hashes presentes
    with open(f'registro_destino_{pack_number}.csv', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            arquivo = row['Arquivo']
            hash_extraido = extrair_hash(arquivo)
            if hash_extraido:
                csv_hashes.add(hash_extraido)
except FileNotFoundError:
    print(f"Erro: Arquivo 'registro_destino_{pack_number}.csv' não encontrado.")
    sys.exit(1)

# Encontrar os hashes que estão no MD5 mas faltam no CSV
missing_hashes = md5_hashes - csv_hashes

# Preparar os dados para salvar no CSV de faltantes
missing = [{'MD5': hash_value} for hash_value in missing_hashes]

# Exibir os resultados
if missing:
    print(f"Hashes presentes no MD5 mas faltando no CSV (Total: {len(missing)}):")
    for item in missing:
        print(f"MD5: {item['MD5']}")
    
    # Salvar os hashes faltantes em um CSV
    nome_arquivo_csv = f'missing_files/missing_files_00{pack_number}.csv'
    try:
        with open(nome_arquivo_csv, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['MD5']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

            writer.writeheader()
            for item in missing:
                writer.writerow(item)
        print(f"\nHashes faltantes foram salvos em '{nome_arquivo_csv}'.")
    except Exception as e:
        print(f"Erro ao escrever no arquivo '{nome_arquivo_csv}': {e}")
else:
    print("Todos os hashes MD5 presentes no arquivo foram encontrados no CSV.")


Hashes presentes no MD5 mas faltando no CSV (Total: 5):
MD5: # virusshare_00469.zip         #
MD5: # twitter: @vxshare            #
MD5: ################################
MD5: # http://virusshare.com        #
MD5: # malware sample md5 list for  #

Hashes faltantes foram salvos em 'missing_files/missing_files_0000469.csv'.
